> 在线讲义：[第 10 章 · 组合、进程内通信与零拷贝](https://wikieden.github.io/ros2-from-scratch/lessons/10-composition-zero-copy.html) · 本书《从零学 ROS 2》

# 第 10 章 · 组合、进程内通信与零拷贝

本章回答：一张 1080p 图像从相机节点传到检测节点，要被复制几次？能不能一次都不复制？"组件"和普通节点有什么不同？Lyrical 新增的 `rosidl::Buffer` 为什么对 GPU 推理很重要？

配套讲义：仓库里 `ros2/notes/lessons/10-composition-zero-copy.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：组件、进程内通信、unique_ptr 都是 C++ 的东西，本机也没有 ROS 2。这里用 Python 的 bytearray 当图像，用 id()（对象身份号，相当于 C++ 里打印的地址）看两边拿到的是不是同一块内存。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 总图 图像大小 | 1920 × 1080 × 3 | `image_bytes()` | 6,220,800 字节 ≈ 6.2 MB |
| 总图 带宽 | × 30 Hz | `size * 30` | 每秒约 187 MB（讲义按 6 MB 取整成 180 MB） |
| 第 2 步 复制 | 普通跨进程 = 复印一份 | `bytes(img) is img` | False |
| 第 2 步 交出 | unique_ptr：递原件，自己不留 | `Handoff().publish_move(...)` | 同一个对象；发布者手里空了 |
| 第 2 步 多订阅者 | 最后一个拿原件，前面的复制 | `deliver(img, 3)` | [False, False, True] |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**类 `Handoff`**：进程内通信的样子：发布者把消息"交出去"（自己的引用清空），订阅者拿到同一个对象。

In [ ]:
class Handoff:
    """进程内通信的样子：发布者把消息"交出去"（自己的引用清空），订阅者拿到同一个对象。"""

    def __init__(self):
        self.msg = None

    def publish_move(self, holder: dict, key: str):
        self.msg = holder.pop(key)  # 相当于 publish(std::move(msg))：交出以后发布者手里就没有了
        return self.msg

**函数 `deliver`**：多个订阅者都要能改的话：最后一个拿原件，前面的各拿一份复制（讲义第 2 步）。

In [ ]:
def deliver(msg: bytearray, n_subscribers: int) -> list[bytearray]:
    """多个订阅者都要能改的话：最后一个拿原件，前面的各拿一份复制（讲义第 2 步）。"""
    return [bytearray(msg) for _ in range(n_subscribers - 1)] + [msg]

**函数 `image_bytes`**：RGB 图像每个像素 3 个字节。

In [ ]:
def image_bytes(width: int = 1920, height: int = 1080, channels: int = 3) -> int:
    """RGB 图像每个像素 3 个字节。"""
    return width * height * channels

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n总图 · 一张 1920×1080 的 RGB 图像有多大")
size = image_bytes()
print(f"  {size:,} 字节 ≈ {size / 1e6:.1f} MB；30 Hz 每秒 {size * 30 / 1e6:.0f} MB")

In [ ]:
print("\n第二步 · 复制 vs 交出：看两边拿到的是不是同一块内存")
img = bytearray(size)
copy = bytes(img)  # 普通跨进程：序列化 / 反序列化，等于复印一份
print("  复印一份：同一个对象吗？", copy is img)
holder = {"msg": img}
received = Handoff().publish_move(holder, "msg")
print("  交出原件：同一个对象吗？", received is img, "；发布者手里还剩", list(holder))

In [ ]:
print("\n第二步（续）· 三个订阅者：前两个各复制一份，最后一个拿原件")
got = deliver(img, 3)
print("  谁拿的是原件:", [g is img for g in got])

## 3. 代码里才看得到的

1. **`id()` / `is` 相当于 C++ 里打印地址**：官方 demo 用两边打印的地址相同来证明零拷贝，Python 里用 `is` 判断是不是同一个对象。
2. **6.2 MB 和 6 MB**：讲义总图按约 6 MB 算出"每秒 180 MB"，精确值是 6.22 MB × 30 ≈ 187 MB。
3. **`holder.pop`** 模仿 `std::move` 之后发布者不能再用原来的指针。发布者之后再访问会怎样，留给练习 10-2。

## 4. 自己改一改

- 把分辨率改成 1280 × 720，算每秒要搬多少 MB；
- 把 `deliver` 的订阅者数改成 1，看是不是一次复制都没有。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **组件**：没有 main 的节点，编成动态库，进容器加载；进程怎么分，部署时再定。
- **进程内**：开 `use_intra_process_comms`，`unique_ptr` 一交，零拷贝。
- **跨进程**：借中间件的共享内存（loaned messages），要定长类型、要中间件支持。
- **GPU**：Lyrical 的 `rosidl::Buffer`，变长数组能直接指向 GPU 内存。